In [1]:
# Cell 0 -- install dependencies (safe to run in Colab or locally;
# quiet flag keeps this from flooding output if already satisfied)
# accelerate>=1.1.0 is REQUIRED by transformers' Trainer/TrainingArguments even though
# this notebook skips training (it still initializes the Trainer for evaluation)
%pip install -q transformers 'accelerate>=1.1.0' torch pandas scikit-learn


Note: you may need to restart the kernel to use updated packages.


# DistilBERT PII Detection

## Goal

Fine-tune DistilBERT for binary classification of privacy-sensitive prompts.

Labels:
- 0 = safe (no PII)
- 1 = privacy sensitive (contains PII)

Evaluation metrics match the classical ML baseline: precision, recall, F1, and confusion matrix per class.

## Setup

This cell works in both Google Colab and a local environment. On Colab, it clones the repo if not already present, then resolves the repo root via `git rev-parse` so every path below is portable regardless of where the notebook is actually running from.


In [2]:
# Cell 1 -- Colab-compatible repo setup
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo_url = "https://github.com/Cyber-207/cyber207_specialized_PII_detection_comparison.git"
    repo_name = "cyber207_specialized_PII_detection_comparison"
    %cd /content
    if not Path(repo_name).exists():
        !git clone {repo_url}
    else:
        print("Repo already cloned.")
    %cd /content/{repo_name}

repo_root = subprocess.run(
    ["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True
).stdout.strip()
repo_root = Path(repo_root)
print("Repo root:", repo_root)

DATA_SPLITS_DIR = repo_root / "data_splits"
RESULTS_DIR = repo_root / "results" / "distilbert"
CHECKPOINT_DIR = RESULTS_DIR / "checkpoint-24408"  # best checkpoint by f1_pii


Repo root: /home/alph9w0lf/Nextcloud/00. Berkeley/06. CYBER 207/__Assignments/Final Project/cyber207_specialized_PII_detection_comparison


In [3]:
# Cell 1b -- environment guards: Git LFS weights + frozen splits
# A clone made without git-lfs leaves a 134-byte POINTER file where the 541 MB checkpoint
# should be, which later fails as "SafetensorError: header too large". Catch it here.
LFS_MAGIC = b"version https://git-lfs"

def is_lfs_pointer(path: Path) -> bool:
    return path.exists() and path.stat().st_size <= 1024 and path.read_bytes().startswith(LFS_MAGIC)

ckpt_weights = CHECKPOINT_DIR / "model.safetensors"
if not ckpt_weights.exists() or is_lfs_pointer(ckpt_weights):
    print("Checkpoint weights missing or LFS pointer -> attempting `git lfs pull` ...")
    if subprocess.run(["git", "lfs", "install"], cwd=repo_root, capture_output=True).returncode == 0:
        subprocess.run(["git", "lfs", "pull"], cwd=repo_root, check=False)
    if not ckpt_weights.exists() or is_lfs_pointer(ckpt_weights):
        raise RuntimeError(
            f"{ckpt_weights} is still a Git LFS pointer (or missing).\n"
            "Fix: install git-lfs, then run `git lfs install && git lfs pull` from the repo root.\n"
            "Verify model.safetensors is ~541 MB, not 134 bytes.")
print(f"Checkpoint weights OK: {ckpt_weights.stat().st_size/1e6:.0f} MB")

split_files = [DATA_SPLITS_DIR / f"{n}.parquet" for n in ("train", "val", "test")]
if not all(p.exists() for p in split_files):
    print("Frozen splits missing -> running src/data/data_split.py ...")
    subprocess.run([sys.executable, str(repo_root / "src/data/data_split.py")], cwd=repo_root, check=True)
print("Frozen splits OK:", ", ".join(p.name for p in split_files))


Checkpoint weights OK: 541 MB
Frozen splits OK: train.parquet, val.parquet, test.parquet


In [4]:
# Cell 2 -- Imports
import pandas as pd
import numpy as np
import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import Dataset

/home/alph9w0lf/Nextcloud/00. Berkeley/06. CYBER 207/__Assignments/Final Project/cyber207_specialized_PII_detection_comparison/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Cell 3 -- confirm GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

Using device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [6]:
# Cell 4 -- load cleaned frozen splits generated by data_split.py
train = pd.read_parquet(DATA_SPLITS_DIR / 'train.parquet')
val = pd.read_parquet(DATA_SPLITS_DIR / 'val.parquet')
test = pd.read_parquet(DATA_SPLITS_DIR / 'test.parquet')

# sanity check
print("train:", train.shape)
print("val:", val.shape)
print("test:", test.shape)


train: (260338, 3)
val: (32542, 3)
test: (32543, 3)


## Tokenization

DistilBERT requires text to be tokenized using its own tokenizer. We use `DistilBertTokenizerFast` with truncation and padding to a max length of **256 tokens**. Token-length analysis during EDA showed a mean of 52.5 tokens (std=24.9), and 256 covers 99.98% of examples in the dataset while keeping compute cost lower than a 512 max length would.


In [7]:
# Cell 6 -- load tokenizer and tokenize splits
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-multilingual-cased')

def tokenize(texts, labels, max_length=256):
    encodings = tokenizer(
        list(texts),
        truncation=True,
        padding=True,
        max_length=max_length
    )
    return encodings, list(labels)

train_encodings, train_labels = tokenize(train['text'], train['label'])
val_encodings, val_labels = tokenize(val['text'], val['label'])
test_encodings, test_labels = tokenize(test['text'], test['label'])

print("Tokenization complete.")

Tokenization complete.


In [8]:
# Cell 7 -- custom dataset class for DistilBERT
class PIIDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = PIIDataset(train_encodings, train_labels)
val_dataset = PIIDataset(val_encodings, val_labels)
test_dataset = PIIDataset(test_encodings, test_labels)

print(f"Train: {len(train_dataset)} examples")
print(f"Val: {len(val_dataset)} examples")
print(f"Test: {len(test_dataset)} examples")

Train: 260338 examples
Val: 32542 examples
Test: 32543 examples


## Model

This notebook loads the already fine-tuned checkpoint (`checkpoint-24408`, selected as the best checkpoint by `f1_pii`) rather than retraining from scratch, so results are reproducible quickly without requiring a multi-epoch training run on whatever hardware this notebook happens to run on. The full training configuration used to originally produce this checkpoint is documented below for transparency.


In [9]:
# Cell 9 -- load the already fine-tuned checkpoint
model = DistilBertForSequenceClassification.from_pretrained(
    str(CHECKPOINT_DIR)
)

model.to(device)
print("Fine-tuned model loaded from:", CHECKPOINT_DIR)
print("Moved to:", device)


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 3546.95it/s]


Fine-tuned model loaded from: /home/alph9w0lf/Nextcloud/00. Berkeley/06. CYBER 207/__Assignments/Final Project/cyber207_specialized_PII_detection_comparison/results/distilbert/checkpoint-24408
Moved to: cuda


In [10]:
# Cell 10 -- define metrics function for Trainer
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    report = classification_report(labels, preds, target_names=['Safe', 'PII'], output_dict=True)
    return {
        'f1_pii': report['PII']['f1-score'],
        'recall_pii': report['PII']['recall'],
        'f1_safe': report['Safe']['f1-score'],
        'f1_macro': report['macro avg']['f1-score'],
        'accuracy': report['accuracy']
    }

In [11]:
# Cell 11 -- configure training arguments
training_args = TrainingArguments(
    output_dir=str(RESULTS_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir=str(RESULTS_DIR / 'logs'),
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_pii',
    fp16=torch.cuda.is_available(),
    seed=42
)


[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [12]:
# Cell 12 -- initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("Trainer initialized.")

Trainer initialized.


## Training Configuration (for reference)

The checkpoint loaded above was originally produced using the configuration below. Evaluation runs after each epoch against the validation split, and the best checkpoint is selected by PII F1 score, since false negatives (sensitive prompts missed) are the primary concern for this project. This notebook does not re-run training, see the Model section above.


## Model Selection Rationale and Remaining Limitations

We use `distilbert-base-multilingual-cased` rather than the English-only `distilbert-base-uncased`. The AI4Privacy dataset is approximately 79% non-English (German, Spanish, French, Italian, Dutch), and switching to the multilingual checkpoint improved PII F1 from 0.973 to 0.983 and reduced false negatives from 624 to 447 in early testing, which is why the multilingual model was selected for the final run.

**Remaining limitations:**
- Masking placeholder tokens (e.g. `[IPV6_1]`) are under-flagged in some cases, contributing to a meaningful share of false negatives. See error analysis below.
- Some false positives stem from ambiguous ground-truth labeling on financial and vehicle identifiers in business-context messages, rather than a clear model weakness.
- Dense numeric strings (long account-like numbers, OTP codes) embedded in business or technical narrative text without a nearby label cue are sometimes missed.


In [13]:
# Cell 14 -- training is skipped; model was already fine-tuned
# and loaded from checkpoint-24408 above. Uncomment the line below to
# retrain from scratch instead (multi-epoch run, GPU time required).

# trainer.train()

print("Skipping training: using pre-trained checkpoint loaded above.")


Skipping training: using pre-trained checkpoint loaded above.


## Evaluation

Evaluate the best checkpoint against the validation set. Metrics match the classical ML baseline: precision, recall, F1 per class, and confusion matrix. Special attention on PII recall -- false negatives represent sensitive prompts incorrectly classified as safe.

In [14]:
# Cell 16 -- evaluate best model on validation set
val_predictions = trainer.predict(val_dataset)
val_preds = val_predictions.predictions.argmax(-1)

print(classification_report(val_labels, val_preds, target_names=['Safe', 'PII']))
print(confusion_matrix(val_labels, val_preds))

              precision    recall  f1-score   support

        Safe       0.96      0.97      0.96     10616
         PII       0.98      0.98      0.98     21926

    accuracy                           0.98     32542
   macro avg       0.97      0.97      0.97     32542
weighted avg       0.98      0.98      0.98     32542

[[10263   353]
 [  409 21517]]


## Test Set Evaluation

Run on the held-out test set only after validation results are satisfactory. Do not use test results to tune the model.

In [15]:
# Cell 18 -- evaluate best model on held-out test set
test_predictions = trainer.predict(test_dataset)
test_preds = test_predictions.predictions.argmax(-1)

print(classification_report(test_labels, test_preds, target_names=['Safe', 'PII']))
print(confusion_matrix(test_labels, test_preds))

              precision    recall  f1-score   support

        Safe       0.96      0.96      0.96     10617
         PII       0.98      0.98      0.98     21926

    accuracy                           0.97     32543
   macro avg       0.97      0.97      0.97     32543
weighted avg       0.97      0.97      0.97     32543

[[10223   394]
 [  447 21479]]


In [16]:
# Export row-level test predictions for cross-model error comparison
test_probs = torch.softmax(torch.tensor(test_predictions.predictions), dim=-1)[:, 1].numpy()

predictions_df = pd.DataFrame({
    "text": test["text"].tolist(),
    "true_label": test_labels,
    "predicted_label": test_preds,
    "pii_confidence": test_probs,
})
predictions_df.to_csv(RESULTS_DIR / "distilbert_test_predictions.csv", index=False)
print(f"Saved {len(predictions_df)} predictions to {RESULTS_DIR / 'distilbert_test_predictions.csv'}")

Saved 32543 predictions to /home/alph9w0lf/Nextcloud/00. Berkeley/06. CYBER 207/__Assignments/Final Project/cyber207_specialized_PII_detection_comparison/results/distilbert/distilbert_test_predictions.csv


In [17]:
# Export confusion-matrix rows (schema mirrors classical/mlp CSVs; test row = canonical)
cm_rows = []
for eval_set, labels, preds, canonical in (("validation", val_labels, val_preds, False),
                                           ("test",       test_labels, test_preds, True)):
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    cm_rows.append({"model": "DistilBERT", "eval_set": eval_set, "operating_point": "default",
                    "canonical": canonical, "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)})
distilbert_cm = pd.DataFrame(cm_rows)
distilbert_cm.to_csv(repo_root / "results" / "distilbert_confusion_matrices.csv", index=False)
print("saved results/distilbert_confusion_matrices.csv"); print(distilbert_cm.to_string(index=False))


saved results/distilbert_confusion_matrices.csv
     model   eval_set operating_point  canonical    tn  fp  fn    tp
DistilBERT validation         default      False 10263 353 409 21517
DistilBERT       test         default       True 10223 394 447 21479


## Final Test Set Results

Evaluated on the full cleaned frozen test split (32,543 rows):

| Metric | Safe | PII | Macro |
|---|---|---|---|
| Precision | 0.96 | 0.98 | — |
| Recall | 0.96 | 0.98 | — |
| F1 | 0.96 | 0.98 | 0.97 |

**Overall accuracy: 0.97.** False positives: 395. False negatives: 447.

The best checkpoint was selected on `f1_pii` rather than accuracy, since class imbalance (67% PII / 33% safe) makes accuracy a misleading selection metric, and false negatives are the primary risk for a PII detector used as a pre-submission screening layer.


## Error Analysis

**False positives (395):** Manual review shows many contain what look like genuinely sensitive identifiers (real-looking account numbers, VIN numbers, bank routing details) that ground truth marks as safe, most often in business-context messages. This points more toward ambiguity in the ground truth labeling for financial/vehicle identifiers in business contexts than a model weakness.

**False negatives (447):** A recurring pattern is masking placeholder tokens (e.g. `[IPV6_1]`) being under-flagged, along with dense numeric strings (long account-like numbers, OTP codes) embedded in business or technical narrative text without an explicit nearby label cue.

**Cross-model note:** this same placeholder-masking artifact also appears in the Phi-4 baseline, but in the opposite direction (Phi-4 over-flags placeholders as PII, DistilBERT under-flags them). This points to a genuine dataset/preprocessing artifact rather than a weakness specific to either model. See `docs/brandon_error_analysis.md` for the full cross-model comparison.
